<img src="https://github.com/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/figs/IOAI-Logo.png?raw=1" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Beijing, China), Individual Contest](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1Rxc_7CEJ0tJJeKibINK6VXH2OU6GtyFd?usp=sharing)

### Data Loading

In [1]:
# Create a shallow sparse clone to download ONLY the Radar directory
!git clone --depth 1 --filter=blob:none --sparse https://github.com/IOAI-official/IOAI-2025.git
%cd IOAI-2025
!git sparse-checkout set Individual-Contest/Radar

# Check the downloaded contents (Task statement, starter notebook, and datasets)
!ls -lh Individual-Contest/Radar

Cloning into 'IOAI-2025'...
remote: Enumerating objects: 179, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (158/158), done.
remote: Total 179 (delta 4), reused 171 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (179/179), 512.60 KiB | 16.02 MiB/s, done.
Resolving deltas: 100% (4/4), done.
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 12.09 KiB | 4.03 MiB/s, done.
/content/IOAI-2025
remote: Enumerating objects: 2809, done.
remote: Counting objects: 100% (2809/2809), done.
remote: Compressing objects: 100% (2808/2808), done.
remote: Total 2809 (delta 2), reused 2807 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (2809/2809), 1.06 GiB | 29.88 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Updating files: 100% (2812/2812), done.
total 100K
drw

In [ ]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class CustomDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)

        images = data[:6]
        labels = data[6]

        images = images.float()
        labels = labels.long()
        labels = labels + 1

        if self.transform:
            images = self.transform(images)
            labels = self.transform(labels)

        return images, labels, self.file_names[idx]

class CustomDataset_test(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)

        images = data[:6]

        images = images.float()

        if self.transform:
            images = self.transform(images)

        return images, self.file_names[idx]

def generate_file_paths(base_path):
    file_paths = []
    for frame in os.listdir(base_path):
        frame_path = os.path.join(base_path, frame)
        if frame_path.endswith('.mat.pt'):
            file_paths.append(frame_path)
    return [path for path in file_paths if os.path.exists(path)]

def load_data(base_path, batch_size=4, num_workers=2, test_size=0.2):
    file_paths = generate_file_paths(base_path)

    train_paths, test_paths = train_test_split(file_paths, test_size=test_size, random_state=42)

    train_dataset = CustomDataset(file_paths=train_paths)
    test_dataset = CustomDataset(file_paths=test_paths)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        drop_last=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        drop_last=True
    )

    return train_loader, test_loader

### Model Definition and Training

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        # Encoder
        self.enc1 = DoubleConv(6, 64)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Bottleneck
        self.bottleneck = DoubleConv(128, 256)

        # Decoder
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128) # 128 from bottleneck + 128 from skip connection

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)   # 64 from decoder + 64 from skip connection

        # Final output layer
        self.final_conv = nn.Conv2d(64, 5, kernel_size=1)

    def forward(self, x):
        # Encoder path (save skip connections)
        s1 = self.enc1(x)
        p1 = self.pool1(s1)

        s2 = self.enc2(p1)
        p2 = self.pool2(s2)

        # Bottleneck
        b = self.bottleneck(p2)

        # Decoder path (concatenate skip connections along channel dim=1)
        d2 = self.up2(b)
        d2 = F.interpolate(d2, size=s2.shape[2:], mode='bilinear', align_corners=False)
        d2 = torch.cat([d2, s2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = F.interpolate(d1, size=s1.shape[2:], mode='bilinear', align_corners=False)
        d1 = torch.cat([d1, s1], dim=1)
        d1 = self.dec1(d1)

        return self.final_conv(d1)

def train(model, train_loader, test_loader, optimizer, criterion, num_epochs=100):
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        batch_count = 0

        for images, labels, _ in train_loader:
            images = images.cuda() if torch.cuda.is_available() else images
            labels = labels.cuda() if torch.cuda.is_available() else labels

            outputs = model(images)
            outputs = outputs.view(outputs.size(0), outputs.size(1), -1)  # [B, C, H*W]
            labels = labels.view(labels.size(0), -1)  # [B, H*W]
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            batch_count += 1

        avg_train_loss = epoch_loss / batch_count
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        val_batch_count = 0

        with torch.no_grad():
            for images, labels, _ in test_loader:
                images = images.cuda() if torch.cuda.is_available() else images
                labels = labels.cuda() if torch.cuda.is_available() else labels

                outputs = model(images)
                outputs = outputs.view(outputs.size(0), outputs.size(1), -1)
                labels = labels.view(labels.size(0), -1)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                val_batch_count += 1

        avg_val_loss = val_loss / val_batch_count
        val_losses.append(avg_val_loss)

        if (epoch+1) % 2 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], '
                  f'Train Loss: {avg_train_loss:.4f}, '
                  f'Val Loss: {avg_val_loss:.4f}')

    return train_losses, val_losses
TRAIN_PATH = "/content/IOAI-2025/Individual-Contest/Radar/"
# The training set is deployed automatically in the testing machine.
# You notebook can access the TRAIN_PATH even if you do not mount it along with notebook.
data_path = TRAIN_PATH + 'training_set'

train_loader, test_loader = load_data(
    base_path=data_path,
    batch_size=4,
    num_workers=2,
    test_size=0.2
)

model = MyModel()
if torch.cuda.is_available():
    model = model.cuda()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses, val_losses = train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=40
)

Epoch [2/40], Train Loss: 0.0316, Val Loss: 0.0249
Epoch [4/40], Train Loss: 0.0202, Val Loss: 0.0200
Epoch [6/40], Train Loss: 0.0175, Val Loss: 0.0190
Epoch [8/40], Train Loss: 0.0159, Val Loss: 0.0176
Epoch [10/40], Train Loss: 0.0139, Val Loss: 0.0172
Epoch [12/40], Train Loss: 0.0122, Val Loss: 0.0181
Epoch [14/40], Train Loss: 0.0105, Val Loss: 0.0181
Epoch [16/40], Train Loss: 0.0086, Val Loss: 0.0194
Epoch [18/40], Train Loss: 0.0070, Val Loss: 0.0208
Epoch [20/40], Train Loss: 0.0056, Val Loss: 0.0226
Epoch [22/40], Train Loss: 0.0047, Val Loss: 0.0230
Epoch [24/40], Train Loss: 0.0032, Val Loss: 0.0286
Epoch [26/40], Train Loss: 0.0037, Val Loss: 0.0289
Epoch [28/40], Train Loss: 0.0027, Val Loss: 0.0308
Epoch [30/40], Train Loss: 0.0017, Val Loss: 0.0337
Epoch [32/40], Train Loss: 0.0031, Val Loss: 0.0285
Epoch [34/40], Train Loss: 0.0014, Val Loss: 0.0361
Epoch [36/40], Train Loss: 0.0024, Val Loss: 0.0279
Epoch [38/40], Train Loss: 0.0012, Val Loss: 0.0352
Epoch [40/40], T

### Generate CSV for Submission

In [6]:
# Run inference on validation set and testing set
from torch.utils.data import DataLoader
import pandas as pd

def run_inference(model, data_loader):
    """Run inference and return predictions with filenames"""
    model.eval()
    predictions = []
    filenames = []

    with torch.no_grad():
        for images, file_names in data_loader:
            images = images.cuda() if torch.cuda.is_available() else images

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            # Convert predictions back to original label range [-1, 3]
            preds = preds - 1

            # Flatten predictions for each sample
            for i, pred in enumerate(preds):
                predictions.append(pred.cpu().numpy().flatten())
                filenames.append(file_names[i])

    return predictions, filenames

#DATA_PATH is the secret environment variable to point the address of the validation set and test set on the testing machine.
#You cannot access this address locally.
if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"
else:
    DATA_PATH = "Solution/"  # Fallback for local testing
# Load validation set
val_paths = generate_file_paths('/content/IOAI-2025/Individual-Contest/Radar/' + DATA_PATH + 'validation_set')
val_dataset = CustomDataset_test(file_paths=val_paths)
val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

# Load testing set
test_paths = generate_file_paths('/content/IOAI-2025/Individual-Contest/Radar/' + DATA_PATH + 'test_set')
test_dataset = CustomDataset_test(file_paths=test_paths)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

# Run inference on validation set
print("Running inference on validation set...")
val_predictions, val_filenames = run_inference(model, val_loader)

# Save validation results to CSV
val_results = []
for filename, pred in zip(val_filenames, val_predictions):
    # Create a row with filename and flattened predictions
    row = {'filename': filename}
    for i, p in enumerate(pred):
        row[f'pixel_{i}'] = p
    val_results.append(row)

val_df = pd.DataFrame(val_results)
val_df.to_csv('submission_val.csv', index=False)
print(f"Validation results saved to output_validation.csv with shape: {val_df.shape}")

# Run inference on testing set
print("Running inference on testing set...")
test_predictions, test_filenames = run_inference(model, test_loader)

# Save testing results to CSV
test_results = []
for filename, pred in zip(test_filenames, test_predictions):
    # Create a row with filename and flattened predictions
    row = {'filename': filename}
    for i, p in enumerate(pred):
        row[f'pixel_{i}'] = p
    test_results.append(row)

test_df = pd.DataFrame(test_results)
test_df.to_csv('submission_test.csv', index=False)
print(f"Testing results saved to output_testing.csv with shape: {test_df.shape}")

print("\nInference completed! Results saved to:")
print("- submission_val.csv (for validation set leaderboard)")
print("- submission_test.csv (for testing set leaderboard)")

Running inference on validation set...
Validation results saved to output_validation.csv with shape: (500, 9051)
Running inference on testing set...
Testing results saved to output_testing.csv with shape: (500, 9051)

Inference completed! Results saved to:
- submission_val.csv (for validation set leaderboard)
- submission_test.csv (for testing set leaderboard)


### Create .zip File

In [7]:
import zipfile
import os

# Define the files to zip and the zip file name.
files_to_zip = ['submission_val.csv', 'submission_test.csv']
zip_filename = 'submission.zip'

# Create a zip file
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # Add the file to the zip fil
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} Created successfully!')

submission.zip Created successfully!


In [8]:
%cd /content

# You need to drag your submission_test.csv, submission_val.csv, ground_truth_test.csv and ground_truth_val.csv to content to get score details

/content


In [9]:
import pandas as pd
import numpy as np

def evaluate_radar_submission(sub_path: str, gt_path: str, bonus: int = 50) -> dict:
    """
    Evaluates submission predictions against ground truth labels based on IOAI Radar rules.

    Parameters:
        sub_path (str): Path to submission CSV file.
        gt_path (str): Path to ground truth CSV file.
        bonus (int): Weight multiplier for correctly predicted non-background pixels (default: 50).

    Returns:
        dict: Detailed breakdown of the points and final normalized score.
    """
    # 1. Load CSV files and set index to filename for fast alignment
    df_sub = pd.read_csv(sub_path).set_index('filename')
    df_gt = pd.read_csv(gt_path).set_index('filename')

    # 2. Align files by common filenames
    common_idx = df_gt.index.intersection(df_sub.index)
    if len(common_idx) < len(df_gt):
        print(f"Warning: {len(df_gt) - len(common_idx)} filenames are missing in {sub_path}!")

    gt_vals = df_gt.loc[common_idx].values
    sub_vals = df_sub.loc[common_idx].values

    # 3. Flatten arrays to evaluate pixel by pixel
    y_true = gt_vals.ravel()
    y_pred = sub_vals.ravel()

    # 4. Create evaluation masks
    is_bg = (y_true == -1)              # Background set C0
    is_non_bg = (y_true != -1)          # Non-background set C1
    is_correct = (y_true == y_pred)     # Correct predictions

    # 5. Calculate correct pixel counts
    c0_total = np.sum(is_bg)
    c1_total = np.sum(is_non_bg)

    c0_correct = np.sum(is_bg & is_correct)
    c1_correct = np.sum(is_non_bg & is_correct)

    # 6. Compute raw points and max possible points
    user_points = (c0_correct * 1) + (c1_correct * bonus)
    max_possible_points = (c0_total * 1) + (c1_total * bonus)

    # 7. Normalize final score (0 - 1)
    score = user_points / max_possible_points if max_possible_points > 0 else 0.0

    return {
        "score": score,
        "user_points": user_points,
        "max_points": max_possible_points,
        "c0_correct": c0_correct,
        "c0_total": c0_total,
        "c0_acc": c0_correct / c0_total if c0_total > 0 else 0.0,
        "c1_correct": c1_correct,
        "c1_total": c1_total,
        "c1_acc": c1_correct / c1_total if c1_total > 0 else 0.0,
    }

# ==========================================
# Run Evaluation for Validation & Test Sets
# ==========================================

if __name__ == "__main__":
    # Evaluate Validation Set
    val_metrics = evaluate_radar_submission('submission_val.csv', 'ground_truth_val.csv')
    print("=== Validation Set Results ===")
    print(f"Normalized Score  : {val_metrics['score']:.6f}")
    print(f"Points Earned     : {val_metrics['user_points']} / {val_metrics['max_points']}")
    print(f"Background Acc    : {val_metrics['c0_correct']}/{val_metrics['c0_total']} ({val_metrics['c0_acc']*100:.2f}%)")
    print(f"Non-Background Acc: {val_metrics['c1_correct']}/{val_metrics['c1_total']} ({val_metrics['c1_acc']*100:.2f}%)\n")

    # Evaluate Test Set
    test_metrics = evaluate_radar_submission('submission_test.csv', 'ground_truth_test.csv')
    print("=== Test Set Results ===")
    print(f"Normalized Score  : {test_metrics['score']:.6f}")
    print(f"Points Earned     : {test_metrics['user_points']} / {test_metrics['max_points']}")
    print(f"Background Acc    : {test_metrics['c0_correct']}/{test_metrics['c0_total']} ({test_metrics['c0_acc']*100:.2f}%)")
    print(f"Non-Background Acc: {test_metrics['c1_correct']}/{test_metrics['c1_total']} ({test_metrics['c1_acc']*100:.2f}%)")

=== Validation Set Results ===
Normalized Score  : 0.909600
Points Earned     : 8923260 / 9810091
Background Acc    : 4404810/4417141 (99.72%)
Non-Background Acc: 90369/107859 (83.78%)

=== Test Set Results ===
Normalized Score  : 0.909975
Points Earned     : 8834329 / 9708318
Background Acc    : 4407429/4419218 (99.73%)
Non-Background Acc: 88538/105782 (83.70%)
